# NORVIA Logo Tampering Detection — Full Pipeline

**Take-home submission — Research Engineer (CV), Monotype Solutions India**

Stage 1 (localize + normalize) → Synthetic labeled benchmark → Stage 2 (6-method ensemble + AUROC calibration + real ranking).

## Setup

- Place raw brief data at:
  - `data/uploads/images/` — the 50 phone photos
  - `data/uploads/reference/` — ref_1.jpg, ref_2.jpg, ref_3.jpg
- First run downloads ~3 GB of models (Grounding DINO, DINOv2, DINO v1, MAE, EasyOCR)
- Total runtime on CUDA GPU: ~15 min

## Excluded from analysis

- 4 test images: 005, 013, 016, 017 — Stage 1 could not reliably capture full logo
- ref_3 — grey-blue decorative band interferes with detection

## 1. Setup

In [ ]:
import os, io, json, warnings
from pathlib import Path
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
import cv2
import torch
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance, ImageFilter
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 90

# --- Paths ---
ROOT       = Path.cwd()
UPLOADS    = ROOT / "data" / "uploads"
REF_DIR    = UPLOADS / "reference"
IMG_DIR    = UPLOADS / "images"

CANONICAL      = ROOT / "data" / "canonical"
CANONICAL_REF  = CANONICAL / "reference"
CANONICAL_IMG  = CANONICAL / "images"

SYNTH_DIR      = ROOT / "data" / "synthetic"
SYNTH_GEN_DIR  = SYNTH_DIR / "genuine"
SYNTH_TAM_DIR  = SYNTH_DIR / "tampered"

OUTPUTS    = ROOT / "outputs"
FIG_DIR    = OUTPUTS / "figures"
LOG_DIR    = OUTPUTS / "scores"

for d in [CANONICAL_REF, CANONICAL_IMG, SYNTH_GEN_DIR, SYNTH_TAM_DIR, FIG_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Config ---
CANONICAL_W, CANONICAL_H = 800, 400
GREEN_THRESHOLD          = 40

# Grounding DINO
GDINO_PROMPT_PRIMARY   = "norvia logo with pure botanicals text."
GDINO_PROMPT_FALLBACK  = "green logo with text."
GDINO_BOX_THRESHOLD    = 0.30
GDINO_TEXT_THRESHOLD   = 0.25
GDINO_LOW_THRESHOLD    = 0.15

# Exclusions
EXCLUDED_TEST = {"005", "013", "016", "017"}
EXCLUDED_REFS = {"ref_3"}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

n_ref = len(list(REF_DIR.glob("*.jpg")))
n_img = len(list(IMG_DIR.glob("*.jpg")))
print(f"Refs found: {n_ref}, Test images found: {n_img}")
assert n_ref > 0 and n_img > 0

In [ ]:
# Clean up any stale canonical files before running
for d in [CANONICAL_REF, CANONICAL_IMG]:
    for f in d.glob("*.jpg"):
        f.unlink()
print("Cleared old canonical crops.")

---

# Stage 1 — Localize + Normalize

Grounding DINO → verify icon + wordmark + tagline via green mask → 4-tier fallback chain → affine rotation → resize 800×400 → CLAHE + channel normalization.

## 2. Load Grounding DINO

In [ ]:
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
import inspect

GDINO_MODEL_ID = "IDEA-Research/grounding-dino-tiny"

print(f"Loading {GDINO_MODEL_ID}...")
gd_processor = AutoProcessor.from_pretrained(GDINO_MODEL_ID)
gd_model = AutoModelForZeroShotObjectDetection.from_pretrained(GDINO_MODEL_ID).to(DEVICE)
gd_model.eval()

_post_sig = inspect.signature(gd_processor.post_process_grounded_object_detection)
_use_new_args = "threshold" in _post_sig.parameters
print(f"Grounding DINO loaded. Uses new API args: {_use_new_args}")


@torch.no_grad()
def gdino_detect(img_rgb, prompt=GDINO_PROMPT_PRIMARY, box_thr=GDINO_BOX_THRESHOLD):
    """Run Grounding DINO. Returns list of (bbox, score) sorted by score descending."""
    pil = Image.fromarray(img_rgb) if isinstance(img_rgb, np.ndarray) else img_rgb
    inputs = gd_processor(images=pil, text=prompt, return_tensors="pt").to(DEVICE)
    outputs = gd_model(**inputs)

    kwargs = dict(target_sizes=[pil.size[::-1]], text_threshold=GDINO_TEXT_THRESHOLD)
    if _use_new_args:
        kwargs["threshold"] = box_thr
    else:
        kwargs["box_threshold"] = box_thr

    results = gd_processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids, **kwargs
    )[0]

    boxes = results["boxes"].cpu().numpy()
    scores = results["scores"].cpu().numpy()

    detections = []
    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = box
        if (x2 - x1) * (y2 - y1) > 500:
            detections.append((tuple(int(v) for v in box), float(scores[i])))

    detections.sort(key=lambda x: -x[1])
    return detections

## 3. Green-mask element detectors (icon / wordmark / tagline)

In [ ]:
def find_green_blobs(img_rgb, min_area=100, kernel_size=(15, 8)):
    """Return list of (x1, y1, x2, y2, area) for green blobs."""
    arr = img_rgb.astype(int)
    green = (arr[..., 1] - arr[..., 0]).astype(np.int16)
    mask = (green > GREEN_THRESHOLD).astype(np.uint8) * 255

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, kernel_size)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    n_labels, _, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)

    blobs = []
    for i in range(1, n_labels):
        x, y, w, h, area = stats[i]
        if area >= min_area:
            blobs.append((int(x), int(y), int(x + w), int(y + h), int(area)))
    return blobs, mask


def detect_wordmark(img_rgb):
    """Find the wordmark — biggest green blob."""
    blobs, _ = find_green_blobs(img_rgb, min_area=300, kernel_size=(20, 10))
    if not blobs:
        return None
    return max(blobs, key=lambda b: b[4])


def detect_icon(img_rgb, wordmark):
    """Find icon — green blob to the LEFT of wordmark, roughly same vertical position."""
    if wordmark is None:
        return None
    wx1, wy1, wx2, wy2, _ = wordmark
    wordmark_h = wy2 - wy1
    wordmark_cy = (wy1 + wy2) / 2

    blobs, _ = find_green_blobs(img_rgb, min_area=100, kernel_size=(10, 10))
    for b in blobs:
        x1, y1, x2, y2, area = b
        if (x1, y1, x2, y2) == wordmark[:4]:
            continue
        cy = (y1 + y2) / 2
        if x2 <= wx1 + 30 and abs(cy - wordmark_cy) < wordmark_h * 0.8:
            return b
    return None


def detect_tagline(img_rgb, wordmark):
    """Find tagline — thin green text BELOW the wordmark."""
    if wordmark is None:
        return None
    wx1, wy1, wx2, wy2, _ = wordmark
    wordmark_h = wy2 - wy1
    img_h = img_rgb.shape[0]

    tagline_search_y_start = wy2 - int(wordmark_h * 0.1)
    tagline_search_y_end = min(img_h, wy2 + int(wordmark_h * 1.5))

    if tagline_search_y_start >= img_h:
        return None

    search_region = img_rgb[tagline_search_y_start:tagline_search_y_end, :]
    if search_region.size == 0:
        return None

    blobs, _ = find_green_blobs(search_region, min_area=50, kernel_size=(25, 5))
    if not blobs:
        return None

    wx_range = (wx1 - 50, wx2 + 50)
    candidates = []
    for b in blobs:
        x1, y1, x2, y2, area = b
        y1_full = y1 + tagline_search_y_start
        y2_full = y2 + tagline_search_y_start
        cx = (x1 + x2) / 2
        if wx_range[0] <= cx <= wx_range[1]:
            candidates.append((x1, y1_full, x2, y2_full, area))

    if not candidates:
        return None
    return max(candidates, key=lambda b: b[4])


def classify_logo_elements(img_rgb):
    wordmark = detect_wordmark(img_rgb)
    icon = detect_icon(img_rgb, wordmark) if wordmark else None
    tagline = detect_tagline(img_rgb, wordmark) if wordmark else None
    return {'icon': icon, 'wordmark': wordmark, 'tagline': tagline}


def verify_full_logo(img_rgb, bbox):
    """Verify bbox contains icon + wordmark + tagline. Returns (quality, elements, reason)."""
    x1, y1, x2, y2 = bbox
    x1 = max(0, x1); y1 = max(0, y1)
    x2 = min(img_rgb.shape[1], x2); y2 = min(img_rgb.shape[0], y2)
    crop = img_rgb[y1:y2, x1:x2]

    if crop.size == 0:
        return 0.0, {}, "empty_crop"

    elements = classify_logo_elements(crop)

    has_icon = elements['icon'] is not None
    has_wordmark = elements['wordmark'] is not None
    has_tagline = elements['tagline'] is not None

    n_elements = sum([has_icon, has_wordmark, has_tagline])
    quality = n_elements / 3.0

    missing = []
    if not has_icon: missing.append("icon")
    if not has_wordmark: missing.append("wordmark")
    if not has_tagline: missing.append("tagline")
    reason = "complete" if quality == 1.0 else f"missing: {', '.join(missing)}"

    return quality, elements, reason

## 4. Bbox expansion + robust detection with fallbacks

In [ ]:
def expand_bbox(bbox, img_shape, missing_elements, expansion_factor=0.5):
    x1, y1, x2, y2 = bbox
    h_img, w_img = img_shape[:2]
    bw = x2 - x1
    bh = y2 - y1
    if 'icon' in missing_elements:
        x1 = max(0, x1 - int(bw * expansion_factor))
    if 'tagline' in missing_elements:
        y2 = min(h_img, y2 + int(bh * expansion_factor))
    return (x1, y1, x2, y2)


def detect_from_wordmark_and_expand(img_rgb):
    """Fallback: detect wordmark, expand by fixed proportions."""
    wordmark = detect_wordmark(img_rgb)
    if wordmark is None:
        return None
    wx1, wy1, wx2, wy2, _ = wordmark
    wordmark_h = wy2 - wy1
    wordmark_w = wx2 - wx1
    img_h, img_w = img_rgb.shape[:2]
    x1 = max(0, wx1 - int(wordmark_w * 0.35))
    y1 = max(0, wy1 - int(wordmark_h * 0.15))
    x2 = min(img_w, wx2 + int(wordmark_w * 0.05))
    y2 = min(img_h, wy2 + int(wordmark_h * 0.6))
    return (x1, y1, x2, y2)


def detect_logo_robust(img_rgb, verbose=False):
    """Main detection with 4-tier fallback. Returns (bbox, quality, method, log)."""
    log = []

    # Attempt 1: Primary GDINO
    detections = gdino_detect(img_rgb, prompt=GDINO_PROMPT_PRIMARY, box_thr=GDINO_BOX_THRESHOLD)
    log.append(f"GDINO primary: {len(detections)} detections")

    for i, (bbox, score) in enumerate(detections[:3]):
        quality, elements, reason = verify_full_logo(img_rgb, bbox)
        log.append(f"  candidate {i}: score={score:.2f}, quality={quality:.2f} ({reason})")

        if quality >= 1.0:
            return bbox, quality, "gdino_primary", log

        if quality >= 0.66:
            missing = [e for e in ['icon', 'wordmark', 'tagline'] if elements.get(e) is None]
            expanded = expand_bbox(bbox, img_rgb.shape, missing)
            q2, e2, r2 = verify_full_logo(img_rgb, expanded)
            log.append(f"    expanded: quality={q2:.2f} ({r2})")
            if q2 >= 1.0:
                return expanded, q2, "gdino_primary_expanded", log

    # Attempt 2: Fallback prompt
    detections = gdino_detect(img_rgb, prompt=GDINO_PROMPT_FALLBACK, box_thr=GDINO_BOX_THRESHOLD)
    log.append(f"GDINO fallback prompt: {len(detections)} detections")

    for i, (bbox, score) in enumerate(detections[:3]):
        quality, elements, reason = verify_full_logo(img_rgb, bbox)
        log.append(f"  candidate {i}: score={score:.2f}, quality={quality:.2f} ({reason})")

        if quality >= 1.0:
            return bbox, quality, "gdino_fallback", log

        if quality >= 0.66:
            missing = [e for e in ['icon', 'wordmark', 'tagline'] if elements.get(e) is None]
            expanded = expand_bbox(bbox, img_rgb.shape, missing)
            q2, _, r2 = verify_full_logo(img_rgb, expanded)
            log.append(f"    expanded: quality={q2:.2f} ({r2})")
            if q2 >= 1.0:
                return expanded, q2, "gdino_fallback_expanded", log

    # Attempt 3: Low-threshold GDINO
    detections = gdino_detect(img_rgb, prompt=GDINO_PROMPT_PRIMARY, box_thr=GDINO_LOW_THRESHOLD)
    log.append(f"GDINO low threshold: {len(detections)} detections")

    best_quality = 0.0
    best_bbox = None
    for i, (bbox, score) in enumerate(detections[:5]):
        quality, elements, reason = verify_full_logo(img_rgb, bbox)
        log.append(f"  candidate {i}: score={score:.2f}, quality={quality:.2f} ({reason})")

        if quality >= 1.0:
            return bbox, quality, "gdino_low_thr", log

        if quality > best_quality:
            best_quality = quality
            best_bbox = bbox

    # Attempt 4: Green-mask direct
    log.append("Falling back to green-mask direct detection")
    blobs, _ = find_green_blobs(img_rgb, min_area=500)
    if blobs:
        xs1 = min(b[0] for b in blobs)
        ys1 = min(b[1] for b in blobs)
        xs2 = max(b[2] for b in blobs)
        ys2 = max(b[3] for b in blobs)
        pad = 20
        h_img, w_img = img_rgb.shape[:2]
        xs1 = max(0, xs1 - pad)
        ys1 = max(0, ys1 - pad)
        xs2 = min(w_img, xs2 + pad)
        ys2 = min(h_img, ys2 + pad)
        bbox = (xs1, ys1, xs2, ys2)
        quality, _, reason = verify_full_logo(img_rgb, bbox)
        log.append(f"  green-mask: quality={quality:.2f} ({reason})")
        if quality >= 0.66:
            return bbox, quality, "green_mask", log

    # Best-effort return
    if best_bbox is not None:
        log.append(f"Returning best-effort GDINO: quality={best_quality:.2f}")
        return best_bbox, best_quality, "gdino_best_effort", log

    return None, 0.0, "failed", log


# Sanity check on ref_1
print("Testing on ref_1...")
img = np.asarray(Image.open(REF_DIR / "ref_1.jpg").convert("RGB"))
bbox, q, method, log = detect_logo_robust(img, verbose=True)
print(f"\nResult: bbox={bbox}, quality={q:.2f}, method={method}")

## 5. Colour normalization + rotation + full Stage 1 pipeline

In [ ]:
def resize_to_canonical(crop_rgb):
    if crop_rgb is None or crop_rgb.size == 0:
        return None
    return cv2.resize(crop_rgb, (CANONICAL_W, CANONICAL_H), interpolation=cv2.INTER_CUBIC)


def apply_clahe(img_rgb):
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    L, a, b = cv2.split(lab)
    L_eq = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(L)
    return cv2.cvtColor(cv2.merge([L_eq, a, b]), cv2.COLOR_LAB2RGB)


REF_MEANS = None
REF_STDS = None


def normalize_channels(img_rgb):
    if REF_MEANS is None:
        return img_rgb
    arr = img_rgb.astype(np.float32)
    om = arr.reshape(-1, 3).mean(axis=0)
    os_ = arr.reshape(-1, 3).std(axis=0) + 1e-6
    arr = (arr - om) / os_
    arr = arr * REF_STDS + REF_MEANS
    return np.clip(arr, 0, 255).astype(np.uint8)


def rotate_via_homography(crop_rgb, angle_threshold_deg=3.0):
    """Rotate if tilted more than threshold. Uses minAreaRect on green mask."""
    if crop_rgb is None or crop_rgb.size == 0:
        return crop_rgb, 0.0

    arr = crop_rgb.astype(int)
    green = (arr[..., 1] - arr[..., 0]).astype(np.int16)
    mask = (green > GREEN_THRESHOLD).astype(np.uint8) * 255
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 12))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return crop_rgb, 0.0

    largest = max(contours, key=cv2.contourArea)
    if cv2.contourArea(largest) < 500:
        return crop_rgb, 0.0

    rect = cv2.minAreaRect(largest)
    (cx, cy), (w, h), angle = rect

    if w < h:
        angle = angle + 90
    while angle > 45:  angle -= 90
    while angle < -45: angle += 90

    if abs(angle) < angle_threshold_deg:
        return crop_rgb, 0.0

    h_img, w_img = crop_rgb.shape[:2]
    center = (w_img // 2, h_img // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    corrected = cv2.warpAffine(crop_rgb, M, (w_img, h_img),
                                flags=cv2.INTER_CUBIC,
                                borderMode=cv2.BORDER_REPLICATE)
    return corrected, float(angle)


def stage1_pipeline(img_rgb):
    """Full Stage 1: detect → crop → rotate → resize → CLAHE → normalize."""
    bbox, quality, method, log = detect_logo_robust(img_rgb)
    if bbox is None or quality < 0.66:
        return None, quality, method, log

    x1, y1, x2, y2 = bbox
    crop = img_rgb[y1:y2, x1:x2]

    rotated, angle = rotate_via_homography(crop, angle_threshold_deg=3.0)
    if abs(angle) > 0.1:
        log.append(f"Rotated by {angle:.1f}°")

    resized = resize_to_canonical(rotated)
    if resized is None:
        return None, quality, method, log

    clahed = apply_clahe(resized)
    normalized = normalize_channels(clahed)
    return normalized, quality, method, log


print("Stage 1 pipeline ready.")

## 6. Process references (compute REF_MEANS / STDS from ref_1)

In [ ]:
ref_paths = [p for p in sorted(REF_DIR.glob("*.jpg")) if p.stem not in EXCLUDED_REFS]
print(f"Reference images: {[p.name for p in ref_paths]}")

# Pass 1: process ref_1 without normalization to compute stats
ref1 = np.asarray(Image.open(REF_DIR / "ref_1.jpg").convert("RGB"))
bbox, q, method, log = detect_logo_robust(ref1)
print(f"ref_1: quality={q:.2f}, method={method}")

if bbox is None:
    raise RuntimeError("Could not detect ref_1 logo — cannot compute reference stats")

x1, y1, x2, y2 = bbox
crop = ref1[y1:y2, x1:x2]
resized = resize_to_canonical(crop)
ref_clahed = apply_clahe(resized)

ref_arr = ref_clahed.astype(np.float32).reshape(-1, 3)
REF_MEANS = ref_arr.mean(axis=0)
REF_STDS = ref_arr.std(axis=0) + 1e-6
print(f"REF_MEANS: {REF_MEANS.round(2)}")
print(f"REF_STDS:  {REF_STDS.round(2)}")

# Pass 2: full pipeline on all refs
for p in ref_paths:
    img = np.asarray(Image.open(p).convert("RGB"))
    canon, q, method, log = stage1_pipeline(img)
    if canon is not None:
        Image.fromarray(canon).save(CANONICAL_REF / p.name, quality=92)
        print(f"  {p.name}: saved, quality={q:.2f}, method={method}")
    else:
        print(f"  {p.name}: FAILED")

# Visualise
fig, axes = plt.subplots(len(ref_paths), 2, figsize=(14, 3 * len(ref_paths)))
if len(ref_paths) == 1: axes = axes.reshape(1, -1)
for i, p in enumerate(ref_paths):
    axes[i, 0].imshow(Image.open(p)); axes[i, 0].set_title(f"{p.name} — raw"); axes[i, 0].axis("off")
    canon_path = CANONICAL_REF / p.name
    if canon_path.exists():
        axes[i, 1].imshow(Image.open(canon_path))
        axes[i, 1].set_title(f"{p.name} — canonical")
        axes[i, 1].axis("off")
plt.tight_layout(); plt.savefig(FIG_DIR / "stage1_refs.png", dpi=100); plt.show()

## 7. Process 46 test images

In [ ]:
real_paths = [p for p in sorted(IMG_DIR.glob("*.jpg")) if p.stem not in EXCLUDED_TEST]
print(f"Processing {len(real_paths)} real images ({len(EXCLUDED_TEST)} excluded: {sorted(EXCLUDED_TEST)})")

results = []
for p in tqdm(real_paths, desc="Stage 1"):
    img = np.asarray(Image.open(p).convert("RGB"))
    canon, quality, method, log = stage1_pipeline(img)

    if canon is not None:
        Image.fromarray(canon).save(CANONICAL_IMG / p.name, quality=92)
        status = "OK"
    else:
        status = "FAILED"

    results.append({
        "name": p.stem,
        "status": status,
        "quality": quality,
        "method": method,
    })

df_stage1 = pd.DataFrame(results)
df_stage1.to_csv(LOG_DIR / "stage1_log.csv", index=False)

print(f"\n=== STAGE 1 SUMMARY ===")
print(f"Total processed: {len(df_stage1)}")
print(f"OK: {(df_stage1['status'] == 'OK').sum()}")
print(f"FAILED: {(df_stage1['status'] == 'FAILED').sum()}")
print(f"\nBy method:")
print(df_stage1['method'].value_counts().to_string())
print(f"\nQuality distribution:")
print(df_stage1['quality'].describe().round(3).to_string())

## 8. Audit grid — all canonical crops

In [ ]:
canon_paths = sorted(CANONICAL_IMG.glob("*.jpg"))
n = len(canon_paths); cols = 5; rows = (n + cols - 1) // cols

quality_map = dict(zip(df_stage1['name'], df_stage1['quality']))
method_map = dict(zip(df_stage1['name'], df_stage1['method']))

fig, axes = plt.subplots(rows, cols, figsize=(cols*4, rows*2.5))
axes = axes.flatten()
for i, p in enumerate(canon_paths):
    axes[i].imshow(Image.open(p))
    q = quality_map.get(p.stem, 0)
    m = method_map.get(p.stem, "?")
    axes[i].set_title(f"{p.stem}  q={q:.2f}  {m}", fontsize=8)
    axes[i].axis("off")
for i in range(len(canon_paths), len(axes)):
    axes[i].axis("off")

plt.suptitle(f"All {n} canonical crops", fontsize=14, y=1.001)
plt.tight_layout(); plt.savefig(FIG_DIR / "stage1_all_canonical.png", dpi=80); plt.show()

---

# Synthetic labeled benchmark

Generate 100 genuines + 240 tampereds (4 types × 3 strengths × 20) from ref_1's CANONICAL crop. Because the source is already Stage-1-processed, synthetics inherit normalization by construction. Only camera effects Stage 1 doesn't fully remove are applied: small tilt, blur, JPEG.

## 9. Synthetic setup

In [ ]:
N_GENUINE = 100
N_TAMPERED_PER_CELL = 20
TAMPER_TYPES = ["stretch", "recolour", "respace", "letter_swap"]
TAMPER_STRENGTHS = ["subtle", "medium", "strong"]
SEED = 42

source_pil = Image.open(CANONICAL_REF / "ref_1.jpg").convert("RGB")
print(f"Synthetic source: ref_1 canonical, size={source_pil.size}")

plt.figure(figsize=(10, 4))
plt.imshow(source_pil); plt.title("Synthetic source (ref_1 canonical)"); plt.axis("off")
plt.tight_layout(); plt.show()

## 10. Camera effect + tamper functions

In [ ]:
def apply_tilt(img, m=3):
    return img.rotate(np.random.uniform(-m, m), resample=Image.BILINEAR, fillcolor=(240, 237, 225))

def apply_blur(img, m=1.2):
    return img.filter(ImageFilter.GaussianBlur(radius=np.random.uniform(0, m)))

def apply_jpeg(img, r=(78, 92)):
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=int(np.random.uniform(*r)))
    buf.seek(0)
    return Image.open(buf).convert("RGB")

def apply_camera_effects(img):
    return apply_jpeg(apply_blur(apply_tilt(img)))


def tamper_stretch(img, s):
    f = {"subtle": 1.06, "medium": 1.18, "strong": 1.35}[s]
    w, h = img.size
    s2 = img.resize((int(w * f), h), Image.BILINEAR)
    return s2.crop(((s2.width - w) // 2, 0, (s2.width - w) // 2 + w, h))


def tamper_recolour(img, s):
    shift = {"subtle": 20, "medium": 45, "strong": 75}[s]
    a = np.asarray(img).astype(int)
    m = (a[..., 1] - a[..., 0]) > 40
    a[m, 0] = np.clip(a[m, 0] + shift, 0, 255)
    a[m, 2] = np.clip(a[m, 2] - shift // 2, 0, 255)
    return Image.fromarray(a.astype(np.uint8))


def tamper_respace(img, s):
    px = {"subtle": 4, "medium": 10, "strong": 20}[s]
    a = np.asarray(img); h, w = a.shape[:2]
    l = a[:, :w//3]
    m = cv2.resize(a[:, w//3:2*w//3], (w//3 + px*2, h))
    r = a[:, 2*w//3:]
    return Image.fromarray(np.concatenate([l, m, r], axis=1)).resize((w, h), Image.BILINEAR)


def tamper_letter_swap(img, s):
    arr = np.asarray(img).copy(); h, w = arr.shape[:2]
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    n_labels, _, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    letters = [(x, y, x+bw, y+bh, area) for i, (x, y, bw, bh, area) in enumerate(stats[1:], 1)
               if area > 500 and y < h * 0.6 and bh > 40 and bw < w * 0.15 and bw > 15]
    if len(letters) < 3:
        return img
    letters.sort(key=lambda l: l[0])
    idx = min(2, len(letters) - 1)
    x1, y1, x2, y2, _ = letters[idx]
    bg = arr[10:30, 10:30].mean(axis=(0, 1)).astype(np.uint8)
    if s == "subtle":
        letter = arr[y1:y2, x1:x2].copy()
        nw, nh = int((x2-x1)*0.85), int((y2-y1)*0.90)
        small = cv2.resize(letter, (nw, nh), interpolation=cv2.INTER_LINEAR)
        arr[y1:y2, x1:x2] = bg
        oy, ox = (y2-y1-nh)//2, (x2-x1-nw)//2
        arr[y1+oy:y1+oy+nh, x1+ox:x1+ox+nw] = small
    elif s == "medium" and idx + 1 < len(letters):
        nx1, ny1, nx2, ny2, _ = letters[idx+1]
        la = arr[y1:y2, x1:x2].copy(); lb = arr[ny1:ny2, nx1:nx2].copy()
        arr[y1:y2, x1:x2] = cv2.resize(lb, (x2-x1, y2-y1))
        arr[ny1:ny2, nx1:nx2] = cv2.resize(la, (nx2-nx1, ny2-ny1))
    else:
        arr[y1:y2, x1:x2] = bg
    return Image.fromarray(arr)


TAMPER_FUNCS = {"stretch": tamper_stretch, "recolour": tamper_recolour,
                "respace": tamper_respace, "letter_swap": tamper_letter_swap}

# Preview one of each at medium
fig, axes = plt.subplots(4, 1, figsize=(10, 8))
for i, (name, fn) in enumerate(TAMPER_FUNCS.items()):
    axes[i].imshow(fn(source_pil.copy(), "medium"))
    axes[i].set_title(f"{name} — medium"); axes[i].axis("off")
plt.tight_layout(); plt.show()

## 11. Generate synthetics (100 genuines + 240 tampereds)

In [ ]:
# Clear old
for d in [SYNTH_GEN_DIR, SYNTH_TAM_DIR]:
    for f in d.glob("*.jpg"):
        f.unlink()

# Genuines
np.random.seed(SEED)
for i in tqdm(range(N_GENUINE), desc="genuine"):
    apply_camera_effects(source_pil.copy()).save(SYNTH_GEN_DIR / f"gen_{i:03d}.jpg", quality=90)

# Tampereds
np.random.seed(SEED + 1)
count = 0
for tt in TAMPER_TYPES:
    for s in TAMPER_STRENGTHS:
        for i in tqdm(range(N_TAMPERED_PER_CELL), desc=f"{tt}_{s}", leave=False):
            t = TAMPER_FUNCS[tt](source_pil.copy(), s)
            apply_camera_effects(t).save(SYNTH_TAM_DIR / f"{tt}_{s}_{i:03d}.jpg", quality=90)
            count += 1

print(f"Generated: {len(list(SYNTH_GEN_DIR.glob('*.jpg')))} genuines, {count} tampereds")

# Verification grid (one sample per combo)
fig, axes = plt.subplots(4, 3, figsize=(15, 12))
for i, tt in enumerate(TAMPER_TYPES):
    for j, s in enumerate(TAMPER_STRENGTHS):
        sp = SYNTH_TAM_DIR / f"{tt}_{s}_000.jpg"
        if sp.exists():
            axes[i, j].imshow(Image.open(sp)); axes[i, j].set_title(f"{tt} — {s}", fontsize=10)
        axes[i, j].axis("off")
plt.tight_layout(); plt.savefig(FIG_DIR / "synth_grid.png", dpi=100); plt.show()

---

# Stage 2 — 6-method ensemble

| # | Method | Signal |
|---|---|---|
| 1 | ORB + RANSAC (rigid) | Geometric changes |
| 2 | SSIM | Pixel structural similarity |
| 3 | OCR vs "NORVIA" | Text substitution |
| 4 | DINOv2 patch-NN | Deep visual anomaly |
| 5 | DINO v1 patch-NN | Diversity backbone |
| 6 | MAE patch-NN | Reconstruction-based SSL |

Patch-level (not CLS token) — CLS distributions overlapped for genuine vs tampered.

## 12. Load reference bank

In [ ]:
REF_CANONICAL_FILES = sorted(CANONICAL_REF.glob("*.jpg"))
reference_bank = {p.stem: np.asarray(Image.open(p).convert("RGB")) for p in REF_CANONICAL_FILES}
print(f"Reference bank: {list(reference_bank.keys())}")

REAL_CANONICAL_FILES = [p for p in sorted(CANONICAL_IMG.glob("*.jpg")) if p.stem not in EXCLUDED_TEST]
SYNTH_GEN_FILES = sorted(SYNTH_GEN_DIR.glob("*.jpg"))
SYNTH_TAM_FILES = sorted(SYNTH_TAM_DIR.glob("*.jpg"))
print(f"Real: {len(REAL_CANONICAL_FILES)}, Genuine synth: {len(SYNTH_GEN_FILES)}, Tampered synth: {len(SYNTH_TAM_FILES)}")

METHOD_NAMES = ["orb", "ssim", "ocr", "dinov2", "dinov1", "mae"]

## 13. Method 1 — ORB + RANSAC (rigid transform)

In [ ]:
orb = cv2.ORB_create(nfeatures=500)
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)


def score_orb(img_rgb, ref_rgb):
    img_gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    ref_gray = cv2.cvtColor(ref_rgb, cv2.COLOR_RGB2GRAY)
    kp1, des1 = orb.detectAndCompute(ref_gray, None)
    kp2, des2 = orb.detectAndCompute(img_gray, None)
    if des1 is None or des2 is None or len(kp1) < 10 or len(kp2) < 10:
        return 1.0

    matches = bf.knnMatch(des1, des2, k=2)
    good = []
    for pair in matches:
        if len(pair) < 2: continue
        m, n = pair
        if m.distance < 0.75 * n.distance:
            good.append(m)
    if len(good) < 10:
        return 1.0

    src = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    try:
        # RIGID transform (rotation + uniform scale + translation)
        # Rejects stretch/shear as anomalous
        _, mask = cv2.estimateAffinePartial2D(src, dst, method=cv2.RANSAC, ransacReprojThreshold=5.0)
        n_inliers = int(mask.sum()) if mask is not None else 0
    except Exception:
        return 1.0
    return 1.0 - (n_inliers / len(good))


def score_orb_vs_bank(img_rgb):
    return min(score_orb(img_rgb, ref) for ref in reference_bank.values())

print("ORB ready.")

## 14. Method 2 — SSIM

In [ ]:
from skimage.metrics import structural_similarity as ssim


def score_ssim(img_rgb, ref_rgb):
    ig = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    rg = cv2.cvtColor(ref_rgb, cv2.COLOR_RGB2GRAY)
    if ig.shape != rg.shape:
        ig = cv2.resize(ig, (rg.shape[1], rg.shape[0]))
    return 1.0 - ssim(ig, rg, data_range=255)


def score_ssim_vs_bank(img_rgb):
    return min(score_ssim(img_rgb, ref) for ref in reference_bank.values())

print("SSIM ready.")

## 15. Method 3 — OCR (EasyOCR)

In [ ]:
import easyocr

ocr_reader = easyocr.Reader(['en'], gpu=(DEVICE == "cuda"), verbose=False)
REF_STRING = "NORVIA"


def score_ocr(img_rgb):
    result = ocr_reader.readtext(img_rgb, detail=0)
    if not result:
        return 1.0
    text = "".join(c for c in "".join(result).upper() if c.isalpha())
    if not text:
        return 1.0
    best = 0.0
    for i in range(max(1, len(text) - len(REF_STRING) + 1)):
        chunk = text[i:i+len(REF_STRING)]
        r = SequenceMatcher(None, REF_STRING, chunk).ratio()
        if r > best: best = r
    return 1.0 - best

print("OCR ready.")

## 16. Methods 4–6 — Deep SSL backbones (DINOv2, DINO v1, MAE)

In [ ]:
from transformers import AutoImageProcessor, AutoModel

DINOV2_ID = "facebook/dinov2-base"
DINOV1_ID = "facebook/dino-vitb16"
MAE_ID    = "facebook/vit-mae-base"

print("Loading DINOv2...")
dinov2_processor = AutoImageProcessor.from_pretrained(DINOV2_ID)
dinov2_model = AutoModel.from_pretrained(DINOV2_ID).to(DEVICE).eval()

print("Loading DINO v1...")
dinov1_processor = AutoImageProcessor.from_pretrained(DINOV1_ID)
dinov1_model = AutoModel.from_pretrained(DINOV1_ID).to(DEVICE).eval()

print("Loading MAE...")
mae_processor = AutoImageProcessor.from_pretrained(MAE_ID)
mae_model = AutoModel.from_pretrained(MAE_ID).to(DEVICE).eval()

print("All 3 loaded.")


@torch.no_grad()
def extract_patches(img_rgb, model, processor):
    pil = Image.fromarray(img_rgb) if isinstance(img_rgb, np.ndarray) else img_rgb
    inputs = processor(images=pil, return_tensors="pt").to(DEVICE)
    outputs = model(**inputs)
    patches = outputs.last_hidden_state[0, 1:, :]   # drop CLS
    return torch.nn.functional.normalize(patches, dim=-1)


def build_memory_bank(model, processor):
    return torch.cat([extract_patches(ref, model, processor) for ref in reference_bank.values()], dim=0)


def patch_nn_score(img_rgb, model, processor, bank, top_frac=0.01):
    tp = extract_patches(img_rgb, model, processor)
    sims = tp @ bank.T
    max_sims, _ = sims.max(dim=1)
    distances = 1.0 - max_sims
    k = max(5, int(top_frac * len(distances)))
    top_k, _ = distances.topk(k)
    return top_k.mean().item()


mem_dinov2 = build_memory_bank(dinov2_model, dinov2_processor)
mem_dinov1 = build_memory_bank(dinov1_model, dinov1_processor)
mem_mae    = build_memory_bank(mae_model, mae_processor)
print(f"Memory banks: DINOv2 {mem_dinov2.shape}, DINO v1 {mem_dinov1.shape}, MAE {mem_mae.shape}")


def score_dinov2_vs_bank(img):
    return patch_nn_score(img, dinov2_model, dinov2_processor, mem_dinov2)

def score_dinov1_vs_bank(img):
    return patch_nn_score(img, dinov1_model, dinov1_processor, mem_dinov1)

def score_mae_vs_bank(img):
    return patch_nn_score(img, mae_model, mae_processor, mem_mae)

## 17. Combined scoring function

In [ ]:
def score_all_methods(img_rgb):
    return {
        "orb":    score_orb_vs_bank(img_rgb),
        "ssim":   score_ssim_vs_bank(img_rgb),
        "ocr":    score_ocr(img_rgb),
        "dinov2": score_dinov2_vs_bank(img_rgb),
        "dinov1": score_dinov1_vs_bank(img_rgb),
        "mae":    score_mae_vs_bank(img_rgb),
    }

## 18. Score 340 synthetics

In [ ]:
gen_rows = []
for p in tqdm(SYNTH_GEN_FILES, desc="genuine"):
    img = np.asarray(Image.open(p).convert("RGB"))
    s = score_all_methods(img)
    s.update({"name": p.stem, "label": "genuine", "tamper_type": None, "strength": None})
    gen_rows.append(s)

tam_rows = []
for p in tqdm(SYNTH_TAM_FILES, desc="tampered"):
    img = np.asarray(Image.open(p).convert("RGB"))
    s = score_all_methods(img)
    parts = p.stem.split("_")
    if parts[0] == "letter" and parts[1] == "swap":
        tt, st = "letter_swap", parts[2]
    else:
        tt, st = parts[0], parts[1]
    s.update({"name": p.stem, "label": "tampered", "tamper_type": tt, "strength": st})
    tam_rows.append(s)

df_synth = pd.DataFrame(gen_rows + tam_rows)
df_synth.to_csv(LOG_DIR / "synthetic_scores.csv", index=False)
print(f"Scored {len(df_synth)} synthetics")
print(df_synth['tamper_type'].value_counts(dropna=False).to_string())

## 19. Score 46 real images

In [ ]:
real_rows = []
for p in tqdm(REAL_CANONICAL_FILES, desc="real"):
    img = np.asarray(Image.open(p).convert("RGB"))
    s = score_all_methods(img)
    s["name"] = p.stem
    real_rows.append(s)

df_real = pd.DataFrame(real_rows)
df_real.to_csv(LOG_DIR / "real_scores.csv", index=False)
print(f"Scored {len(df_real)} real images")
print(df_real[METHOD_NAMES].describe().round(3).to_string())

## 20. AUROC — per method × tamper type

In [ ]:
from sklearn.metrics import roc_auc_score

labels_binary = (df_synth['label'] == 'tampered').astype(int)
auroc_results = {m: roc_auc_score(labels_binary, df_synth[m]) for m in METHOD_NAMES}

print("=== Overall AUROC ===")
for m, a in sorted(auroc_results.items(), key=lambda x: -x[1]):
    print(f"  {m}: {a:.3f}")

# Per-tamper-type
auroc_matrix = {}
for tt in TAMPER_TYPES:
    subset = df_synth[(df_synth['label']=='genuine') | (df_synth['tamper_type']==tt)]
    labels = (subset['label']=='tampered').astype(int)
    auroc_matrix[tt] = {m: roc_auc_score(labels, subset[m]) for m in METHOD_NAMES}

df_auroc = pd.DataFrame(auroc_matrix).T[METHOD_NAMES]
print("\n=== AUROC by tamper type ===")
print(df_auroc.round(3).to_string())
df_auroc.to_csv(LOG_DIR / "auroc_by_tamper.csv")

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
ms = sorted(auroc_results.items(), key=lambda x: -x[1])
axes[0].barh([m[0] for m in ms], [m[1] for m in ms], color='steelblue')
axes[0].axvline(0.5, color='red', linestyle='--', alpha=0.5, label='random')
axes[0].set_xlabel("AUROC"); axes[0].set_title("Overall AUROC")
axes[0].set_xlim(0, 1.05); axes[0].legend()

im = axes[1].imshow(df_auroc.values, cmap='RdYlGn', vmin=0.5, vmax=1.0, aspect='auto')
axes[1].set_xticks(range(len(METHOD_NAMES))); axes[1].set_xticklabels(METHOD_NAMES, rotation=45, ha='right')
axes[1].set_yticks(range(len(TAMPER_TYPES))); axes[1].set_yticklabels(TAMPER_TYPES)
axes[1].set_title("AUROC per method × tamper type")
for i in range(len(TAMPER_TYPES)):
    for j in range(len(METHOD_NAMES)):
        v = df_auroc.values[i, j]
        axes[1].text(j, i, f"{v:.2f}", ha='center', va='center',
                     color='white' if v < 0.7 else 'black', fontsize=10)
plt.colorbar(im, ax=axes[1], label='AUROC')
plt.tight_layout(); plt.savefig(FIG_DIR / "auroc.png", dpi=100); plt.show()

## 21. Rank real images

Distribution-mismatch check: synthetic-genuine mean vs real mean per method.
Synthetic thresholds don't transfer to real (documented in report §3), so AUROC just weights the methods; ranking uses real-relative z-scores.

In [ ]:
print("Distribution comparison — synthetic-genuine vs real:")
print(f"{'method':<10} {'synth-gen mean':<15} {'real mean':<12} {'ratio':<8}")
for m in METHOD_NAMES:
    sg = df_synth[df_synth['label']=='genuine'][m].mean()
    r = df_real[m].mean()
    ratio = r / (sg + 1e-6)
    print(f"{m:<10} {sg:<15.3f} {r:<12.3f} {ratio:<8.1f}x")

# Weights from AUROC (0.5 → 0 weight, 1.0 → 1)
weights = {m: max(0, (auroc_results[m] - 0.5) * 2) for m in METHOD_NAMES}
total = sum(weights.values())
weights = {m: w / total for m, w in weights.items()}
print("\nMethod weights (AUROC-derived):")
for m, w in sorted(weights.items(), key=lambda x: -x[1]):
    print(f"  {m}: {w:.3f}")

# Rank against REAL distribution
real_stats = {m: {"mean": df_real[m].mean(), "std": df_real[m].std() + 1e-6} for m in METHOD_NAMES}
df_real_r = df_real.copy()
for m in METHOD_NAMES:
    df_real_r[f"{m}_z"] = (df_real[m] - real_stats[m]["mean"]) / real_stats[m]["std"]
df_real_r['ensemble_z'] = sum(df_real_r[f"{m}_z"] * weights[m] for m in METHOD_NAMES)

df_ranked = df_real_r.sort_values('ensemble_z', ascending=False).reset_index(drop=True)
df_ranked['rank'] = df_ranked.index + 1
df_ranked.to_csv(LOG_DIR / "real_ranked.csv", index=False)

print("\n=== TOP 12 MOST SUSPICIOUS ===")
print(df_ranked[['rank', 'name', 'ensemble_z'] + METHOD_NAMES].head(12).round(3).to_string(index=False))

## 22. Top-12 canonical crops

In [ ]:
top12 = df_ranked.head(12)
fig, axes = plt.subplots(4, 3, figsize=(15, 10))
axes = axes.flatten()
for i, (_, row) in enumerate(top12.iterrows()):
    p = CANONICAL_IMG / f"{row['name']}.jpg"
    if p.exists():
        axes[i].imshow(Image.open(p))
        axes[i].set_title(f"#{int(row['rank'])} — {row['name']} (z={row['ensemble_z']:.2f})", fontsize=10)
    axes[i].axis("off")
plt.suptitle("Top 12 most suspicious real images", fontsize=13, y=1.001)
plt.tight_layout(); plt.savefig(FIG_DIR / "top12.png", dpi=100); plt.show()

# Where's 009?
r009 = df_ranked[df_ranked['name']=='009']
if len(r009):
    r = r009.iloc[0]
    print(f"\nImage 009 (known letter_swap tamper): rank {int(r['rank'])}/{len(df_ranked)}, ensemble_z={r['ensemble_z']:.3f}")

---

# Done

Files produced:
- `data/canonical/` — Stage 1 outputs (46 test + 2 refs)
- `data/synthetic/` — 100 genuines + 240 tampereds
- `outputs/scores/` — CSVs (stage1_log, synthetic_scores, real_scores, auroc_by_tamper, real_ranked)
- `outputs/figures/` — visualisations

See `report.md` for interpretation, limitations, and next steps.